# PathRAG Full-System Colab Run

Run this notebook top-to-bottom on a **GPU runtime**. Use an A100 or similar if you want local `llava-med` to be stable.

This notebook does four things:
1. clones `pathrag-agentic-starter`
2. creates dedicated tool environments for histocartography and CHIEF with **Python 3.9**
3. installs a local `llava-med` runtime in Colab
4. runs smoke checks and then a **full graph run**

What you still need to provide:
- a slide or flat pathology image path
- CHIEF source tree
- CHIEF model weights
- Hugging Face access to `microsoft/llava-med-v1.5-mistral-7b`
- optional `OPENAI_API_KEY` if you want real Stage 5 / Stage 7

## 0. Runtime choice

Before running the notebook:
- `Runtime` -> `Change runtime type`
- Hardware accelerator: `GPU`
- Preferred: `A100`

The notebook keeps the **main orchestration** in the Colab kernel and isolates these heavy tools in separate environments:
- histocartography: Python 3.9
- CHIEF: Python 3.9
- LLaVA-Med: Python 3.10

In [ ]:
from pathlib import Path
import os

# Edit these before running the rest of the notebook.
BRANCH = "local-chief"
REPO_URL = "https://github.com/YOUR-ORG/path-agent.git"
WORKDIR = Path("/content/path-agent")
STARTER_DIR = WORKDIR / "pathrag-agentic-starter"

# Input image + question for the full graph run.
# Use a flat image (.jpg/.png) or a slide path (.svs).
IMAGE_PATH = "/content/drive/MyDrive/pathrag_inputs/07_level2.jpg"
QUESTION = "What are the prominent, discrete, thick-walled circular structures with clear or collapsed lumens embedded within the looser pale pink stroma?"
TOP_K = 3
MAX_ROUNDS = 1

# Where CHIEF source + weights live in Drive.
DRIVE_CHIEF_REPO_DIR = "/content/drive/MyDrive/pathrag_assets/CHIEF"
DRIVE_CHIEF_MODEL_DIR = "/content/drive/MyDrive/pathrag_assets/CHIEF/model_weight"

# LLaVA-Med setup.
LLAVA_REPO_URL = "https://github.com/microsoft/LLaVA-Med.git"
LLAVA_REPO_DIR = Path("/content/third_party/LLaVA-Med")
LLAVA_MODEL = "microsoft/llava-med-v1.5-mistral-7b"
LLAVA_CONV_MODE = "mistral_instruct"

# Optional: Stage 5 / Stage 7 real OpenAI path.
OPENAI_API_KEY = ""

for key, value in {
    "BRANCH": BRANCH,
    "REPO_URL": REPO_URL,
    "WORKDIR": str(WORKDIR),
    "STARTER_DIR": str(STARTER_DIR),
    "IMAGE_PATH": IMAGE_PATH,
    "QUESTION": QUESTION,
    "TOP_K": str(TOP_K),
    "MAX_ROUNDS": str(MAX_ROUNDS),
    "DRIVE_CHIEF_REPO_DIR": DRIVE_CHIEF_REPO_DIR,
    "DRIVE_CHIEF_MODEL_DIR": DRIVE_CHIEF_MODEL_DIR,
    "LLAVA_REPO_URL": LLAVA_REPO_URL,
    "LLAVA_REPO_DIR": str(LLAVA_REPO_DIR),
    "LLAVA_MODEL": LLAVA_MODEL,
    "LLAVA_CONV_MODE": LLAVA_CONV_MODE,
}.items():
    os.environ[key] = value

print({
    "branch": BRANCH,
    "repo_url": REPO_URL,
    "image_path": IMAGE_PATH,
    "llava_model": LLAVA_MODEL,
})


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
%%bash
set -euo pipefail

apt-get update -y
apt-get install -y git git-lfs rsync curl wget build-essential libgl1 libglib2.0-0 libopenslide0 openslide-tools

if [ ! -x /content/bin/micromamba ]; then
  mkdir -p /content/bin
  cd /content
  curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba
fi

/content/bin/micromamba --help >/dev/null

In [ ]:
%%bash
set -euo pipefail

rm -rf /content/path-agent
mkdir -p /content

git clone "$REPO_URL" /content/path-agent
cd /content/path-agent
git checkout "$BRANCH"

In [ ]:
import pathlib
import subprocess
import sys

starter_reqs = STARTER_DIR / "requirements.txt"
filtered = []
for line in starter_reqs.read_text(encoding="utf-8").splitlines():
    raw = line.strip()
    if not raw:
        filtered.append(line)
        continue
    if raw.startswith("--extra-index-url"):
        continue
    if raw.startswith("torch"):
        continue
    if raw.startswith("torchvision"):
        continue
    filtered.append(line)

main_req = pathlib.Path("/content/pathrag-main-requirements.txt")
main_req.write_text("\n".join(filtered) + "\n", encoding="utf-8")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(main_req)])
print("Main runtime ready")


In [ ]:
%%bash
set -euo pipefail

mkdir -p ~/.local/share/pathrag/chief
mkdir -p ~/.local/share/pathrag/histocartography/checkpoints
rm -rf ~/.local/share/pathrag/chief/repo ~/.local/share/pathrag/chief/model_weight
mkdir -p ~/.local/share/pathrag/chief/repo ~/.local/share/pathrag/chief/model_weight

if [ ! -d "$DRIVE_CHIEF_REPO_DIR" ]; then
  echo "Missing CHIEF repo dir: $DRIVE_CHIEF_REPO_DIR" >&2
  exit 1
fi
if [ ! -d "$DRIVE_CHIEF_MODEL_DIR" ]; then
  echo "Missing CHIEF model dir: $DRIVE_CHIEF_MODEL_DIR" >&2
  exit 1
fi

rsync -a "$DRIVE_CHIEF_REPO_DIR"/ ~/.local/share/pathrag/chief/repo/
rsync -a "$DRIVE_CHIEF_MODEL_DIR"/ ~/.local/share/pathrag/chief/model_weight/

ls -la ~/.local/share/pathrag/chief
ls -la ~/.local/share/pathrag/chief/model_weight

In [ ]:
%%bash
set -euo pipefail

MAMBA=/content/bin/micromamba
HC_ENV=/content/envs/histocartography39
CHIEF_ENV=/content/envs/chief39
LLAVA_ENV=/content/envs/llava310

$MAMBA create -y -p "$HC_ENV" python=3.9 pip
$MAMBA create -y -p "$CHIEF_ENV" python=3.9 pip
$MAMBA create -y -p "$LLAVA_ENV" python=3.10 pip

$MAMBA run -p "$HC_ENV" python --version
$MAMBA run -p "$CHIEF_ENV" python --version
$MAMBA run -p "$LLAVA_ENV" python --version

In [ ]:
%%bash
set -euo pipefail

MAMBA=/content/bin/micromamba
HC_ENV=/content/envs/histocartography39
CHIEF_ENV=/content/envs/chief39
LLAVA_ENV=/content/envs/llava310

# GPU-enabled torch stacks for the tool envs.
$MAMBA run -p "$HC_ENV" pip install -q --index-url https://download.pytorch.org/whl/cu124 torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1
$MAMBA run -p "$CHIEF_ENV" pip install -q --index-url https://download.pytorch.org/whl/cu124 torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1
$MAMBA run -p "$LLAVA_ENV" pip install -q --index-url https://download.pytorch.org/whl/cu124 torch torchvision torchaudio

# Tool-specific deps.
$MAMBA run -p "$HC_ENV" pip install -q -r /content/path-agent/pathrag-agentic-starter/tools/histocartography/requirements.txt
$MAMBA run -p "$CHIEF_ENV" pip install -q -r /content/path-agent/pathrag-agentic-starter/tools/chief/requirements.txt timm==0.5.4 rich typer httpx huggingface_hub
$MAMBA run -p "$LLAVA_ENV" pip install -q -r /content/path-agent/pathrag-agentic-starter/tools/llava_med/requirements.txt

In [ ]:
%%bash
set -euo pipefail

if [ ! -d "$LLAVA_REPO_DIR/.git" ]; then
  rm -rf "$LLAVA_REPO_DIR"
  mkdir -p "$(dirname "$LLAVA_REPO_DIR")"
  git clone "$LLAVA_REPO_URL" "$LLAVA_REPO_DIR"
fi

if [ -f "$LLAVA_REPO_DIR/requirements.txt" ]; then
  /content/bin/micromamba run -p /content/envs/llava310 pip install -q -r "$LLAVA_REPO_DIR/requirements.txt"
fi


In [ ]:
import getpass
import os
import subprocess

hf_token = getpass.getpass("Hugging Face token for LLaVA-Med (required if gated): ")
if hf_token:
    subprocess.check_call([
        "/content/bin/micromamba", "run", "-p", "/content/envs/llava310",
        "huggingface-cli", "login", "--token", hf_token,
    ])

if not OPENAI_API_KEY:
    OPENAI_API_KEY = getpass.getpass("OpenAI API key (optional, press Enter to skip): ")

if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [ ]:
import os

os.environ["PYTHONPATH"] = str(STARTER_DIR / "src")
os.environ["HISTOCARTOGRAPHY_PYTHON"] = "/content/envs/histocartography39/bin/python"
os.environ["CHIEF_PYTHON"] = "/content/envs/chief39/bin/python"
os.environ["PATHRAG_STAGE4_BACKEND"] = "llava-med"
os.environ.pop("PATHRAG_STAGE4_REMOTE_URL", None)
os.environ["USE_LLAVA"] = "1"
os.environ["LLMED_REPO"] = str(LLAVA_REPO_DIR)
os.environ["LLMED_PYTHON"] = "/content/envs/llava310/bin/python"
os.environ["LLMED_MODEL"] = LLAVA_MODEL
os.environ["LLMED_CONV_MODE"] = LLAVA_CONV_MODE
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128")

for key in [
    "HISTOCARTOGRAPHY_PYTHON",
    "CHIEF_PYTHON",
    "PATHRAG_STAGE4_BACKEND",
    "USE_LLAVA",
    "LLMED_REPO",
    "LLMED_PYTHON",
    "LLMED_MODEL",
]:
    print(f"{key}={os.environ[key]}")

In [ ]:
import json
import os
import subprocess
from pathlib import Path

hc_cmd = [
    os.environ["HISTOCARTOGRAPHY_PYTHON"],
    "-m", "src.graph.hc_eval",
    "--config", str(STARTER_DIR / "tools" / "histocartography" / "config" / "default.yaml"),
    "--image-path", IMAGE_PATH,
    "--top-n", str(TOP_K),
    "--out", "/content/hc_smoke.json",
]
subprocess.run(hc_cmd, cwd=str(STARTER_DIR / "tools" / "histocartography"), check=True)
print(Path("/content/hc_smoke.json").read_text())


In [ ]:
import json
import os
import subprocess
from pathlib import Path

hc = json.loads(Path("/content/hc_smoke.json").read_text())
valid_bounds = [
    [p["x1"], p["y1"], p["x2"], p["y2"]]
    for p in hc.get("selected_patches", [])[:TOP_K]
]

chief_cmd = [
    os.environ["CHIEF_PYTHON"],
    "-m", "src.graph.chief_eval",
    "--config", str(STARTER_DIR / "tools" / "chief" / "config" / "default.yaml"),
    "--image-path", IMAGE_PATH,
    "--top-k", str(TOP_K),
    "--valid-bounds-json", json.dumps(valid_bounds),
    "--out", "/content/chief_smoke.json",
]
subprocess.run(chief_cmd, cwd=str(STARTER_DIR / "tools" / "chief"), check=True)
print(Path("/content/chief_smoke.json").read_text()[:2000])


In [ ]:
import json
from pathlib import Path
import sys

sys.path.insert(0, str(STARTER_DIR / "src"))
from pathrag.agents.langgraph_app import build_graph

app = build_graph()
init = {
    "image_path": IMAGE_PATH,
    "question": QUESTION,
    "mode": "answer",
    "top_k": TOP_K,
    "max_rounds": MAX_ROUNDS,
    "round_ix": 0,
}

merged = {}
for state in app.stream(init, config={"configurable": {"thread_id": "colab-full-run"}}):
    if isinstance(state, dict):
        merged.update(state)

out_path = Path("/content/pathrag_full_run.json")
out_path.write_text(json.dumps(merged, indent=2), encoding="utf-8")
print(f"Saved full run to {out_path}")

In [ ]:
import json
from pathlib import Path

result = json.loads(Path("/content/pathrag_full_run.json").read_text())

stage12 = result.get("tile_rank", result)
stage3 = result.get("identify", result)
stage4 = result.get("stage4", result)
stage6 = result.get("rerank", result)
stage7 = result.get("fuse", result)

print("Final answer:\n")
print(stage7.get("final_answer", result.get("final_answer", "")))
print("\nHC selected patches:")
print(json.dumps(stage12.get("hc_rank", result.get("hc_rank", [])), indent=2)[:2000])
print("\nCHIEF patches:")
print(json.dumps(stage12.get("patches", result.get("patches", [])), indent=2)[:2000])
print("\nChosen idx:", stage6.get("chosen_idx", result.get("chosen_idx", [])))
print("\nROI descriptions:")
print(json.dumps(stage4.get("roi_desc", result.get("roi_desc", [])), indent=2)[:2000])
print("\nPatch summaries:")
print(json.dumps(stage4.get("patch_summaries", result.get("patch_summaries", [])), indent=2)[:3000])


## Notes

- If the full run fails in Stage 4, debug the `llava-med` environment first. The most common issues are Hugging Face access, model OOM, or an incompatible repo checkout.
- If Stage 1 fails, check that:
  - `~/.local/share/pathrag/chief/repo` contains the CHIEF source tree
  - `~/.local/share/pathrag/chief/model_weight` contains `CHIEF_CTransPath.pth`, `CHIEF_pretraining.pth`, and `Text_emdding.pth`
- Histocartography checkpoints download automatically on first use into `~/.local/share/pathrag/histocartography/checkpoints`.